# publications.csv から「oare_id → 候補PDF/ページ」を引く（FTS全文検索）

目的:
- `train.csv` の各 `oare_id` について、`publications.csv`（PDF名×ページ単位OCRテキスト）を全文検索し、候補 `pdf_name/page` を上位3件出す。
- **取り違え/途中欠落/数値表記ゆれ**の修正作業で「原典の当たり」を付けるための探索支援。

注意:
- 初回は `publications.csv`（216k行）から SQLite FTS5 インデックスを作るため時間がかかります（DBは `data/index/` に作成、gitには入れない想定）。
- 検索精度はOCR品質・表記揺れ・原PDFに文字列が含まれるかに依存します。**最終確定は人間**で行ってください。

関連スクリプト:
- `scripts/publications_candidate_search.py`


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# このノートブックは notebooks/EDA/ 配下にある想定
REPO_ROOT = Path.cwd().resolve().parents[1]
SCRIPTS_DIR = REPO_ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

import pandas as pd

import publications_candidate_search as pcs

TRAIN_CSV = REPO_ROOT / "data/kaggle/deep-past-initiative-machine-translation/train.csv"
PUBLISHED_TEXTS_CSV = REPO_ROOT / "data/kaggle/deep-past-initiative-machine-translation/published_texts.csv"
PUBLICATIONS_CSV = REPO_ROOT / "data/kaggle/deep-past-initiative-machine-translation/publications.csv"
DB_PATH = REPO_ROOT / "data/index/publications_fts.sqlite"
CONFIRMED_PATH = REPO_ROOT / "data/index/confirmed_oare_to_publication.csv"

TRAIN_CSV, PUBLISHED_TEXTS_CSV, PUBLICATIONS_CSV

In [ ]:
# 1) FTSインデックス作成（初回のみ重い）
#   - 既にDBがあり publications.csv が同一ならスキップされます

pcs.ensure_publications_fts(
    publications_csv=PUBLICATIONS_CSV,
    db_path=DB_PATH,
    chunk_size=5000,
    rebuild=False,
)

DB_PATH

In [ ]:
# 2) 対象IDの一覧（trainの1561件）

train = pd.read_csv(TRAIN_CSV, usecols=["oare_id", "transliteration", "translation"])
pt = pd.read_csv(PUBLISHED_TEXTS_CSV, usecols=["oare_id", "label", "cdli_id", "online transcript"])
merged = train.merge(pt, on="oare_id", how="left")

print("train", train.shape, "published_texts", pt.shape, "merged", merged.shape)
merged.head(2)

In [ ]:
def show_candidates(oare_id: str, topk: int = 3, max_terms: int = 12):
    row = merged.loc[merged["oare_id"] == oare_id].iloc[0]
    print("oare_id:", row["oare_id"])
    print("label:", row.get("label"))
    print("cdli_id:", row.get("cdli_id"))
    print("online transcript:", row.get("online transcript"))
    print("-" * 80)
    print("translation:")
    print(str(row.get("translation", ""))[:800])
    print("-" * 80)

    df = pcs.generate_candidates(
        train_csv=TRAIN_CSV,
        published_texts_csv=PUBLISHED_TEXTS_CSV,
        db_path=DB_PATH,
        oare_id=oare_id,
        topk=topk,
        max_terms=max_terms,
    )
    display(df[["rank", "pdf_name", "page", "score", "query", "snippet"]])
    return df


# 例: まずは1件試す
example_oare_id = merged["oare_id"].iloc[0]
df_example = show_candidates(example_oare_id, topk=3)
df_example.head(3)

In [ ]:
# 3) 確定した対応（oare_id → pdf_name/page）をローカルCSVに追記していく

from datetime import datetime


def append_confirmation(oare_id: str, pdf_name: str, page: int, note: str = ""):
    CONFIRMED_PATH.parent.mkdir(parents=True, exist_ok=True)
    row = {
        "oare_id": oare_id,
        "pdf_name": pdf_name,
        "page": int(page),
        "note": note,
        "confirmed_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    if CONFIRMED_PATH.exists():
        df = pd.read_csv(CONFIRMED_PATH)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])
    df.to_csv(CONFIRMED_PATH, index=False)
    return row


CONFIRMED_PATH

In [ ]:
# 4) ipywidgets が使える場合の簡易UI（なければこのセルはスキップでOK）

try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    oare_dropdown = widgets.Dropdown(
        options=merged["oare_id"].tolist(),
        description="oare_id",
        layout=widgets.Layout(width="80%"),
    )
    topk_slider = widgets.IntSlider(value=3, min=1, max=10, step=1, description="topk")
    search_btn = widgets.Button(description="検索", button_style="primary")
    out = widgets.Output()

    candidate_choice = widgets.Dropdown(description="確定", options=[("(未選択)", None)])
    note_text = widgets.Text(description="メモ", layout=widgets.Layout(width="80%"))
    confirm_btn = widgets.Button(description="保存", button_style="success")

    state = {"last_df": None}

    def on_search(_):
        with out:
            clear_output(wait=True)
            df = show_candidates(oare_dropdown.value, topk=int(topk_slider.value))
            state["last_df"] = df
            opts = [("(未選択)", None)]
            for _, r in df.dropna(subset=["rank"]).iterrows():
                key = f"#{int(r['rank'])}: {r['pdf_name']} p{int(r['page'])}"
                opts.append((key, (r["pdf_name"], int(r["page"]))))
            candidate_choice.options = opts

    def on_confirm(_):
        choice = candidate_choice.value
        if choice is None:
            with out:
                print("[WARN] 候補が未選択です")
            return
        pdf_name, page = choice
        row = append_confirmation(oare_dropdown.value, pdf_name, page, note=note_text.value)
        with out:
            print("[OK] saved:", row)

    search_btn.on_click(on_search)
    confirm_btn.on_click(on_confirm)

    display(widgets.VBox([widgets.HBox([oare_dropdown, topk_slider, search_btn]), out]))
    display(widgets.HBox([candidate_choice, note_text, confirm_btn]))

except Exception as e:
    print("ipywidgets が使えませんでした:", repr(e))
    print("上の show_candidates() と append_confirmation() を手動で呼び出してください")